In [12]:
import sys
from pathlib import Path
import pandas as pd

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


def ler_intervalos(path_xlsx: str, intervalo_minutos: int = 15) -> pd.DataFrame:
    """
    Lê um ficheiro Excel no formato da E-Redes e devolve apenas os registos de intervalos,
    com timestamp e energia_kwh calculada. Sem agregações.
    """
    path = Path(path_xlsx)
    df_raw = pd.read_excel(path, header=None)

    # localizar linha do cabeçalho (célula 'Data' na 1ª coluna)
    mask_header = df_raw.iloc[:, 0].astype(str).str.strip().str.lower().eq("data")
    if not mask_header.any():
        raise ValueError(f"[{path.name}] Cabeçalho 'Data' não encontrado.")
    header_row = mask_header.idxmax()
    cols = df_raw.iloc[header_row].tolist()

    # construir df com cabeçalho
    df = df_raw.iloc[header_row + 1:].copy()
    df.columns = cols

    # manter linhas válidas
    df = df[~df["Data"].isna() & ~df["Hora"].isna()].copy()

    # detectar coluna de consumo (qualquer que contenha 'consumo')
    consumo_col_candidates = [c for c in df.columns if isinstance(c, str) and ("consumo" in c.lower())]
    if not consumo_col_candidates:
        raise ValueError(f"[{path.name}] Coluna de consumo não encontrada.")
    consumo_col = consumo_col_candidates[0]

    # normalizar números (vírgula decimal)

    s = (
        df[consumo_col].astype(str).str.strip()
        .replace({"-": pd.NA, "—": pd.NA, "–": pd.NA}, regex=False)  # dashes -> NA
        .str.replace(",", ".", regex=False)
    )
    
    s = pd.to_numeric(s, errors="coerce").ffill()  # usa o valor anterior
    df[consumo_col] = s
    


    

    # timestamp e energia (kW * horas)
    df["timestamp"] = pd.to_datetime(df["Data"].astype(str) + " " + df["Hora"].astype(str), errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()

    # anota o ficheiro/“mês” (ex.: c01, c02, …)
    df["ficheiro"] = path.stem

    # ordena por tempo
    return df.sort_values("timestamp").reset_index(drop=True)

def juntar_intervalos(diretorio: str, saida_xlsx: str = "consumos_juntos.xlsx", padrao: str = "c*.xlsx"):
    base = Path(diretorio)
    files = sorted(base.glob(padrao))
    if not files:
        raise FileNotFoundError(f"Nenhum ficheiro encontrado em {base} com padrão {padrao}")

    todos = []
    for f in files:
        try:
            df = ler_intervalos(str(f), intervalo_minutos=15)
            todos.append(df)
            print(f"[OK] {f.name}: {len(df)} registos")
        except Exception as e:
            print(f"[ERRO] {f.name}: {e}")

    if not todos:
        raise RuntimeError("Nenhum ficheiro foi processado com sucesso.")

    df_all = pd.concat(todos, ignore_index=True)

    df_all = df_all.drop(columns=["timestamp"])


    

    # escreve um único XLSX, uma única folha
    out = Path(saida_xlsx)
    with pd.ExcelWriter(out, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm", date_format="yyyy-mm-dd") as xlw:
        df_all.to_excel(xlw, index=False, sheet_name="intervalos")

    print(f"\n✅ Ficheiro gerado: {out.resolve()}")
    print(f"   Total de registos: {len(df_all)}")

    return df_all


diretorio = 'data'
saida =  "merged_consumos.xlsx"
juntar_intervalos(diretorio, saida_xlsx=saida, padrao="c*.xlsx")


[OK] c01.xlsx: 2976 registos
[OK] c02.xlsx: 2688 registos
[OK] c03.xlsx: 2972 registos
[OK] c04.xlsx: 2880 registos
[OK] c05.xlsx: 2976 registos
[OK] c06.xlsx: 2880 registos
[OK] c07.xlsx: 2976 registos
[OK] c08.xlsx: 2976 registos
[OK] c09.xlsx: 1638 registos

✅ Ficheiro gerado: /Users/miguelferreira/Desktop/SLB/ElectricitySpotPrices/Simulation/nb/merged_consumos.xlsx
   Total de registos: 24962


,Data,Hora,Consumo registado (kW),Estado,ficheiro
0,2025/01/01,00:15,0.564,Real,c01
1,2025/01/01,00:30,0.480,Real,c01
2,2025/01/01,00:45,0.336,Real,c01
3,2025/01/01,01:00,0.440,Real,c01
4,2025/01/01,01:15,0.428,Real,c01
...,...,...,...,...,...
24957,2025/09/18,00:30,0.432,Real,c09
24958,2025/09/18,00:45,0.500,Real,c09
24959,2025/09/18,01:00,0.360,Real,c09
24960,2025/09/18,01:15,0.308,Real,c09
